# Irene AI-generated 2.5D avatar asset

This notebook uses AI background removal, depth estimation, and pose landmarks to build a layered avatar package. It creates an inpainted background plus transparent subject, torso, head, arms, legs, and depth layers.

The local Irene app automatically uses the package after it is copied to `avatar/live/irene_2p5d/`. A free Colab GPU is helpful but not guaranteed.

In [ ]:
!pip -q install "rembg[gpu]" transformers accelerate mediapipe opencv-python-headless pillow

In [ ]:
from google.colab import files

print('Upload Irene photo (PNG or JPG).')
uploaded = files.upload()
source_name = next(iter(uploaded))
print('Uploaded:', source_name)

In [ ]:
import json
import shutil
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import torch
from PIL import Image
from rembg import remove
from transformers import pipeline

source = Image.open('/content/' + source_name).convert('RGB')
max_dim = 1024
scale = min(1.0, max_dim / max(source.size))
if scale < 1:
    source = source.resize((round(source.width * scale), round(source.height * scale)), Image.Resampling.LANCZOS)
width, height = source.size
print('Working size:', width, 'x', height)
print('GPU available:', torch.cuda.is_available())

In [ ]:
print('Creating AI subject cutout...')
subject_rgba = remove(source)
if subject_rgba.mode != 'RGBA':
    subject_rgba = subject_rgba.convert('RGBA')
alpha = np.array(subject_rgba.getchannel('A'))

print('Estimating depth...')
depth_pipe = pipeline(
    'depth-estimation',
    model='depth-anything/Depth-Anything-V2-Small-hf',
    device=0 if torch.cuda.is_available() else -1,
)
depth_image = depth_pipe(source)['depth'].resize((width, height), Image.Resampling.BILINEAR)
depth = np.array(depth_image).astype(np.float32)
depth = ((depth - depth.min()) / max(depth.max() - depth.min(), 1e-6) * 255).astype(np.uint8)
print('AI cutout and depth map ready.')

In [ ]:
print('Detecting body joints...')
mp_pose = mp.solutions.pose
with mp_pose.Pose(static_image_mode=True, model_complexity=1, enable_segmentation=False) as pose:
    pose_result = pose.process(np.array(source))
if not pose_result.pose_landmarks:
    raise RuntimeError('No body pose was detected. Use a photo where the full body is visible.')

landmarks = pose_result.pose_landmarks.landmark
def xy(index):
    point = landmarks[index]
    return (int(np.clip(point.x, 0, 1) * (width - 1)), int(np.clip(point.y, 0, 1) * (height - 1)))

def line_mask(pairs, thickness):
    mask = np.zeros((height, width), dtype=np.uint8)
    for first, second in pairs:
        a, b = xy(first), xy(second)
        cv2.line(mask, a, b, 255, thickness, cv2.LINE_AA)
        cv2.circle(mask, a, thickness // 2, 255, -1, cv2.LINE_AA)
        cv2.circle(mask, b, thickness // 2, 255, -1, cv2.LINE_AA)
    return mask

body_width = max(24, int(abs(xy(11)[0] - xy(12)[0]) * 0.65))
torso_points = np.array([xy(i) for i in (11, 12, 24, 23)], dtype=np.int32)
torso_mask = np.zeros((height, width), dtype=np.uint8)
cv2.fillConvexPoly(torso_mask, torso_points, 255)

left_arm = line_mask([(11, 13), (13, 15)], max(12, body_width // 3))
right_arm = line_mask([(12, 14), (14, 16)], max(12, body_width // 3))
legs = line_mask([(23, 25), (25, 27), (24, 26), (26, 28)], max(16, body_width // 2))

nose = xy(0)
head_mask = np.zeros((height, width), dtype=np.uint8)
cv2.ellipse(head_mask, nose, (max(18, body_width // 2), max(24, body_width * 2 // 3)), 0, 0, 360, 255, -1)
person_mask = torso_mask.copy()
for layer_mask in (head_mask, left_arm, right_arm, legs):
    person_mask = cv2.bitwise_or(person_mask, layer_mask)
person_mask = cv2.dilate(person_mask, np.ones((17, 17), dtype=np.uint8), iterations=1)
person_mask = cv2.bitwise_and(person_mask, alpha)
print('Body layers created.')

In [ ]:
out_dir = Path('/content/irene_2p5d')
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True)

source_rgba = np.array(source_rgba)
source_bgr = cv2.cvtColor(np.array(source), cv2.COLOR_RGB2BGR)
subject_mask = person_mask
background_bgr = cv2.inpaint(source_bgr, subject_mask, 7, cv2.INPAINT_TELEA)
background = Image.fromarray(cv2.cvtColor(background_bgr, cv2.COLOR_BGR2RGB))
background.save(out_dir / 'background.png')

def save_layer(name, mask):
    layer = source_rgba.copy()
    layer[:, :, 3] = cv2.bitwise_and(mask, person_mask)
    Image.fromarray(layer).save(out_dir / name)

source_rgba[:, :, 3] = person_mask
Image.fromarray(source_rgba).save(out_dir / 'subject.png')
save_layer('torso.png', torso_mask)
save_layer('head.png', head_mask)
save_layer('left_arm.png', left_arm)
save_layer('right_arm.png', right_arm)
save_layer('legs.png', legs)
Image.fromarray(depth).save(out_dir / 'depth.png')
(out_dir / 'manifest.json').write_text(json.dumps({
    'width': width,
    'height': height,
    'source_model': 'rembg + Depth-Anything-V2-Small + MediaPipe Pose',
    'layers': ['background.png', 'subject.png', 'torso.png', 'head.png', 'left_arm.png', 'right_arm.png', 'legs.png', 'depth.png']
}, indent=2))
print('Saved AI 2.5D package:', out_dir)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

preview = Image.open(out_dir / 'background.png').convert('RGBA')
preview.alpha_composite(Image.open(out_dir / 'subject.png'))
display(preview.resize((min(768, preview.width), round(preview.height * min(768, preview.width) / preview.width))))

archive = shutil.make_archive('/content/irene_2p5d', 'zip', '/content', 'irene_2p5d')
print('Download this ZIP and unzip its irene_2p5d folder into your project at avatar/live/.')

### Optional AI background redraw

If Colab assigned a GPU, the next cell can replace the simple inpainted background with a generated office background. Leave it disabled if the free runtime runs out of memory.

In [ ]:
USE_AI_REDRAW = torch.cuda.is_available()
if USE_AI_REDRAW:
    !pip -q install diffusers
    from diffusers import AutoPipelineForInpainting
    redraw_pipe = AutoPipelineForInpainting.from_pretrained(
        'runwayml/stable-diffusion-inpainting',
        torch_dtype=torch.float16,
    ).to('cuda')
    redraw_mask = Image.fromarray(subject_mask).resize(source.size)
    generated_background = redraw_pipe(
        prompt='a clean detailed modern creative studio office, warm natural lighting, empty room, realistic photography',
        negative_prompt='person, face, body, hands, extra limbs, text, watermark',
        image=source,
        mask_image=redraw_mask,
        guidance_scale=7.0,
        num_inference_steps=20,
    ).images[0].convert('RGB')
    generated_background.save(out_dir / 'background.png')
    print('AI-redrawn background saved.')
else:
    print('GPU not available; keeping the fast inpainted background.')

archive = shutil.make_archive('/content/irene_2p5d', 'zip', '/content', 'irene_2p5d')
print('Updated ZIP:', archive)

In [ ]:
from google.colab import files
files.download(archive)